# Singing neighborhoods from shared leaders

This notebook builds "singing neighborhoods" from the people who lead songs at different singings. The basic idea is that two singings are probably connected if many of the same leaders appear at both of them over time.

Each node in the graph is a canonical singing name from `Singings.csv`, not a single minutes record. For example, many yearly records for the same convention can be grouped into one long-running singing identity. A singing participates only if at least `MIN_MAPPED_MINUTES` minutes records map to it, which keeps the analysis from leaning too hard on places with very little data.

The method has three big steps:

1. Collect the set of leaders associated with each canonical singing.
2. Compare singings by how much their leader sets overlap, giving extra weight to leaders who are more locally distinctive.
3. Build a pruned graph from the strongest mutual ties, then cluster connected singings into provisional neighborhoods.

These neighborhoods are data-driven guesses, not official boundaries. They are useful because they summarize patterns of shared participation, but they should still be checked against geography, history, and local knowledge.


In [ ]:
from collections import defaultdict, deque
from itertools import combinations
from pathlib import Path
import os
import sqlite3

os.environ.setdefault("MPLCONFIGDIR", str(Path(".jupyter/matplotlib-cache").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

## Parameters

This cell holds the knobs that control the analysis. Changing them changes which singings qualify, which leader overlaps are considered strong enough, and how small regions are merged.

The most important support filter is `MIN_MAPPED_MINUTES`: a singing needs at least this many mapped minutes records before it participates. Similarity is based on IDF-weighted shared leaders, which means a leader who appears at only a few singings counts as stronger evidence than a leader who appears almost everywhere. The mutual-nearest-neighbor settings keep the graph focused on each singing's strongest relationships instead of letting every weak connection pile into one huge region.

A good way to use this notebook is to adjust one parameter at a time, rerun the later cells, and inspect whether the resulting neighborhoods look historically and geographically plausible.


In [ ]:
DB_PATH = Path("minutes_pre95.db")
SINGINGS_CSV = Path("Singings.csv")
REGION_NAMES_CSV = Path("singing_neighborhood_region_names.csv")

MIN_MAPPED_MINUTES = 5
ONLY_DENSON_MINUTES = True

# Only use leaders with this many total lessons or more.
MIN_LEADER_LESSON_COUNT = 10

# Candidate edge thresholds.
MIN_SHARED_LEADERS = 10
MIN_JACCARD = 0.0
MIN_WEIGHTED_JACCARD = 0.23

# Keep only each singing's strongest neighbors before clustering.
NEIGHBORS_PER_SINGING = 3
REQUIRE_MUTUAL_NEIGHBORS = True
EDGE_SCORE_COLUMN = "weighted_jaccard"

# Rescue otherwise-isolated singings with one strong best edge.
ALLOW_ISOLATE_BEST_EDGE = True
ISOLATE_MIN_WEIGHTED_JACCARD = 0.28

# Exclude leaders who appear in more than this many minutes records.
# Set to None to include all leaders who pass MIN_LEADER_LESSON_COUNT.
MAX_LEADER_MINUTES_COUNT = 300

# Merge small seed regions after strict graph clustering.
MERGE_SMALL_REGIONS = True
MERGE_REGIONS_SMALLER_THAN = 4
MIN_REGION_MERGE_WEIGHTED_JACCARD = 0.15
MAX_MERGED_REGION_SIZE = 20

# Display/export controls.
TOP_EDGE_COUNT = 5
MAX_HEATMAP_MEMBERS = 60
EXPORT_CSVS = True


## Load the Singing Mapping

`Singings.csv` is the bridge between individual minutes records and long-running singing identities. It maps each minutes record ID to a canonical `corrected_singing` name. If `corrected_singing` is blank, the original `singing` value is used.

This step matters because the same event can be written slightly differently across years. Without a canonical mapping, the notebook might treat spelling variants or yearly versions of the same singing as unrelated places.


In [ ]:
singing_rows = pd.read_csv(SINGINGS_CSV)

singing_rows["canonical_singing"] = (
    singing_rows["corrected_singing"]
    .fillna(singing_rows["singing"])
    .astype(str)
    .str.strip()
)

singing_rows = singing_rows[
    singing_rows["id"].notna()
    & singing_rows["canonical_singing"].notna()
    & singing_rows["canonical_singing"].ne("")
    & singing_rows["canonical_singing"].ne("nan")
].copy()
singing_rows["minutes_id"] = singing_rows["id"].astype(int)

mapped_minutes = singing_rows[["minutes_id", "canonical_singing"]].drop_duplicates()

with sqlite3.connect(DB_PATH) as conn:
    mapped_minute_years = pd.read_sql_query(
        "select id as minutes_id, Year as mapped_year, IsDenson as is_denson from minutes",
        conn,
    )

mapped_minutes = mapped_minutes.merge(mapped_minute_years, on="minutes_id", how="left")
if ONLY_DENSON_MINUTES:
    mapped_minutes = mapped_minutes[mapped_minutes["is_denson"].eq(1)].copy()

mapped_support = (
    mapped_minutes.groupby("canonical_singing")
    .agg(
        mapped_minutes=("minutes_id", "nunique"),
        first_year=("mapped_year", "min"),
        last_year=("mapped_year", "max"),
    )
    .sort_values("mapped_minutes", ascending=False)
)

eligible_names_by_mapping = mapped_support[mapped_support["mapped_minutes"] >= MIN_MAPPED_MINUTES].index
mapped_minutes = mapped_minutes[mapped_minutes["canonical_singing"].isin(eligible_names_by_mapping)].copy()

print(f"rows in Singings.csv: {len(singing_rows):,}")
print(f"mapped Denson-only minutes rows: {len(mapped_minutes):,}" if ONLY_DENSON_MINUTES else f"mapped minutes rows: {len(mapped_minutes):,}")
print(f"canonical singings in mapping: {mapped_support.shape[0]:,}")
print(f"canonical singings with at least {MIN_MAPPED_MINUTES} mapped minutes: {len(eligible_names_by_mapping):,}")
mapped_support.head(10)

## Load Leaders And Aggregate By Canonical Singing

This step gathers all leaders connected to each canonical singing. Instead of treating every minutes record separately, it combines records that belong to the same mapped singing identity.

The result is a leader set for each singing: essentially, "these are the people who have appeared as leaders at this singing across the mapped records." Leaders are counted once per canonical singing for overlap purposes, even if they led many lessons there. That keeps the comparison focused on shared participation rather than the number of times one active person led at one event.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    leader_occurrences = pd.read_sql_query(
        """
        select distinct
            slj.minutes_id,
            slj.leader_id,
            l.name as leader_name,
            l.lesson_count as leader_lesson_count,
            m.Name as minutes_name,
            m.Location as location,
            m.Year as year,
            m.Date as date,
            m.IsDenson as is_denson
        from song_leader_joins slj
        join leaders l on l.id = slj.leader_id
        join minutes m on m.id = slj.minutes_id
        where slj.leader_id is not null
        """,
        conn,
    )

if ONLY_DENSON_MINUTES:
    leader_occurrences = leader_occurrences[leader_occurrences["is_denson"].eq(1)].copy()

leader_occurrences = leader_occurrences[leader_occurrences["leader_lesson_count"] >= MIN_LEADER_LESSON_COUNT].copy()

leader_minutes_count = leader_occurrences.groupby("leader_id")["minutes_id"].nunique()
if MAX_LEADER_MINUTES_COUNT is not None:
    leaders_to_keep = leader_minutes_count[leader_minutes_count <= MAX_LEADER_MINUTES_COUNT].index
    leader_occurrences = leader_occurrences[leader_occurrences["leader_id"].isin(leaders_to_keep)].copy()

attendance = leader_occurrences.merge(mapped_minutes, on="minutes_id", how="inner")
attendance = attendance.drop_duplicates(["canonical_singing", "minutes_id", "leader_id"])

singing_leaders = attendance.drop_duplicates(["canonical_singing", "leader_id"])

singing_summary = (
    attendance.groupby("canonical_singing")
    .agg(
        mapped_minutes_with_filtered_leaders=("minutes_id", "nunique"),
        distinct_leaders=("leader_id", "nunique"),
        first_leader_year=("year", "min"),
        last_leader_year=("year", "max"),
    )
    .join(mapped_support, how="left")
    .sort_values(["mapped_minutes", "distinct_leaders"], ascending=False)
)

eligible_singings = singing_summary[singing_summary["mapped_minutes"] >= MIN_MAPPED_MINUTES].copy()

print(f"leader/minutes rows after leader filters: {len(leader_occurrences):,}")
print(f"mapped leader rows: {len(attendance):,}")
print(f"eligible singing nodes: {len(eligible_singings):,}")
print(f"distinct leaders represented in eligible nodes: {singing_leaders['leader_id'].nunique():,}")
eligible_singings.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

eligible_singings["mapped_minutes"].hist(ax=axes[0], bins=30)
axes[0].set_title("Mapped minutes per eligible singing")
axes[0].set_xlabel("mapped minutes")

eligible_singings["distinct_leaders"].hist(ax=axes[1], bins=30)
axes[1].set_title("Distinct leaders per eligible singing")
axes[1].set_xlabel("leaders")

eligible_singings[["mapped_minutes", "distinct_leaders"]].plot.scatter(
    x="mapped_minutes",
    y="distinct_leaders",
    ax=axes[2],
    alpha=0.7,
)
axes[2].set_title("Support vs leader set size")

plt.tight_layout()

## Build Weighted Mutual-Nearest-Neighbor Overlaps

This section creates the edges of the graph. An edge is a possible relationship between two singings based on shared leaders. The stronger the overlap, the stronger the evidence that the two singings belong near each other in a neighborhood.

Several similarity measures are calculated:

- `shared_leaders`: raw count of leaders seen at both canonical singings.
- `jaccard`: shared leaders divided by all leaders seen at either singing.
- `weighted_jaccard`: the same idea, but each shared leader is weighted by inverse document frequency across eligible singings. Locally distinctive leaders count more than leaders who appear almost everywhere.
- `overlap_coefficient`: shared leaders divided by the smaller singing's leader set.

After scoring candidate pairs, the notebook keeps only each singing's top `NEIGHBORS_PER_SINGING` neighbors. With `REQUIRE_MUTUAL_NEIGHBORS = True`, an edge survives only if both singings rank each other highly. This is like saying a friendship tie is stronger evidence when both sides choose each other, not just one side.


In [ ]:
leader_sets = {
    singing: frozenset(group["leader_id"].dropna().astype(int))
    for singing, group in singing_leaders[
        singing_leaders["canonical_singing"].isin(eligible_singings.index)
    ].groupby("canonical_singing")
}

singings_by_leader = (
    singing_leaders[singing_leaders["canonical_singing"].isin(leader_sets)]
    .drop_duplicates(["leader_id", "canonical_singing"])
    .groupby("leader_id")["canonical_singing"]
    .apply(lambda values: sorted(map(str, values)))
)

total_singings = len(leader_sets)
leader_document_frequency = singings_by_leader.apply(len)
leader_weights = np.log((1 + total_singings) / (1 + leader_document_frequency)) + 1
leader_weights = leader_weights.to_dict()
leader_set_weights = {
    singing: sum(leader_weights.get(leader_id, 0.0) for leader_id in leaders)
    for singing, leaders in leader_sets.items()
}

shared_counts = defaultdict(int)
shared_weight_sums = defaultdict(float)
for leader_id, singings in singings_by_leader.items():
    if len(singings) < 2:
        continue
    leader_weight = leader_weights[leader_id]
    for left, right in combinations(singings, 2):
        shared_counts[(left, right)] += 1
        shared_weight_sums[(left, right)] += leader_weight

scored_rows = []
for (left, right), shared in shared_counts.items():
    if shared < MIN_SHARED_LEADERS:
        continue
    left_set = leader_sets[left]
    right_set = leader_sets[right]
    union_size = len(left_set | right_set)
    smaller_size = min(len(left_set), len(right_set))
    jaccard = shared / union_size if union_size else 0
    overlap_coefficient = shared / smaller_size if smaller_size else 0
    shared_weight = shared_weight_sums[(left, right)]
    weighted_union = leader_set_weights[left] + leader_set_weights[right] - shared_weight
    weighted_jaccard = shared_weight / weighted_union if weighted_union else 0
    left_stats = eligible_singings.loc[left]
    right_stats = eligible_singings.loc[right]
    scored_rows.append(
        {
            "singing_a": left,
            "singing_b": right,
            "mapped_minutes_a": int(left_stats["mapped_minutes"]),
            "mapped_minutes_b": int(right_stats["mapped_minutes"]),
            "leaders_a": len(left_set),
            "leaders_b": len(right_set),
            "shared_leaders": shared,
            "shared_leader_weight": shared_weight,
            "jaccard": jaccard,
            "weighted_jaccard": weighted_jaccard,
            "overlap_coefficient": overlap_coefficient,
            "first_year_a": int(left_stats["first_year"]) if pd.notna(left_stats["first_year"]) else np.nan,
            "last_year_a": int(left_stats["last_year"]) if pd.notna(left_stats["last_year"]) else np.nan,
            "first_year_b": int(right_stats["first_year"]) if pd.notna(right_stats["first_year"]) else np.nan,
            "last_year_b": int(right_stats["last_year"]) if pd.notna(right_stats["last_year"]) else np.nan,
        }
    )

all_scored_edges = pd.DataFrame(scored_rows)
if len(all_scored_edges):
    candidate_edges = all_scored_edges[
        (all_scored_edges["jaccard"] >= MIN_JACCARD)
        & (all_scored_edges["weighted_jaccard"] >= MIN_WEIGHTED_JACCARD)
    ].copy()
else:
    candidate_edges = pd.DataFrame(columns=["singing_a", "singing_b", EDGE_SCORE_COLUMN, "shared_leaders"])

if len(candidate_edges):
    candidate_edges = candidate_edges.sort_values([EDGE_SCORE_COLUMN, "shared_leaders"], ascending=False).reset_index(drop=True)

    directed_a = candidate_edges.assign(source= candidate_edges["singing_a"], target=candidate_edges["singing_b"])
    directed_b = candidate_edges.assign(source= candidate_edges["singing_b"], target=candidate_edges["singing_a"])
    directed_neighbors = pd.concat([directed_a, directed_b], ignore_index=True)
    directed_neighbors = directed_neighbors.sort_values(
        ["source", EDGE_SCORE_COLUMN, "shared_leaders"],
        ascending=[True, False, False],
    )
    top_neighbors = directed_neighbors.groupby("source", group_keys=False).head(NEIGHBORS_PER_SINGING)

    directed_pairs = set(zip(top_neighbors["source"], top_neighbors["target"]))
    if REQUIRE_MUTUAL_NEIGHBORS:
        kept_pairs = {tuple(sorted((left, right))) for left, right in directed_pairs if (right, left) in directed_pairs}
    else:
        kept_pairs = {tuple(sorted((left, right))) for left, right in directed_pairs}

    pair_keys = candidate_edges.apply(lambda row: tuple(sorted((row["singing_a"], row["singing_b"]))), axis=1)
    edges = candidate_edges[pair_keys.isin(kept_pairs)].copy()
    edges["edge_source"] = "mutual_top_neighbor"

    fallback_edges = pd.DataFrame()
    if ALLOW_ISOLATE_BEST_EDGE:
        connected_nodes = set(edges["singing_a"]).union(edges["singing_b"]) if len(edges) else set()
        isolated_nodes = set(eligible_singings.index) - connected_nodes
        fallback_candidates = candidate_edges[
            (candidate_edges[EDGE_SCORE_COLUMN] >= ISOLATE_MIN_WEIGHTED_JACCARD)
            & (candidate_edges["singing_a"].isin(isolated_nodes) | candidate_edges["singing_b"].isin(isolated_nodes))
        ].copy()
        if len(fallback_candidates):
            fallback_directed_a = fallback_candidates.assign(
                isolated_node=fallback_candidates["singing_a"],
                other_node=fallback_candidates["singing_b"],
            )
            fallback_directed_b = fallback_candidates.assign(
                isolated_node=fallback_candidates["singing_b"],
                other_node=fallback_candidates["singing_a"],
            )
            fallback_directed = pd.concat([fallback_directed_a, fallback_directed_b], ignore_index=True)
            fallback_directed = fallback_directed[fallback_directed["isolated_node"].isin(isolated_nodes)]
            fallback_directed = fallback_directed.sort_values(
                ["isolated_node", EDGE_SCORE_COLUMN, "shared_leaders"],
                ascending=[True, False, False],
            )
            fallback_edges = fallback_directed.groupby("isolated_node", group_keys=False).head(1)
            fallback_edges = fallback_edges[candidate_edges.columns].drop_duplicates(["singing_a", "singing_b"]).copy()
            fallback_edges["edge_source"] = "isolated_best_edge"
            if len(fallback_edges):
                edges = pd.concat([edges, fallback_edges], ignore_index=True)

    edges = edges.sort_values([EDGE_SCORE_COLUMN, "shared_leaders"], ascending=False).reset_index(drop=True)
else:
    candidate_edges = pd.DataFrame(columns=["singing_a", "singing_b", EDGE_SCORE_COLUMN, "shared_leaders"])
    edges = candidate_edges.copy()
    fallback_edges = pd.DataFrame()

print(f"candidate singing pairs sharing at least one leader: {len(shared_counts):,}")
print(f"scored edges with at least {MIN_SHARED_LEADERS} shared leaders: {len(all_scored_edges):,}")
print(f"candidate edges passing score thresholds: {len(candidate_edges):,}")
print(f"edges after top-{NEIGHBORS_PER_SINGING} {'mutual ' if REQUIRE_MUTUAL_NEIGHBORS else ''}neighbor pruning: {len(edges) - len(fallback_edges):,}")
print(f"isolated-best fallback edges added: {len(fallback_edges):,}")
print(f"total kept edges: {len(edges):,}")
edges.head(TOP_EDGE_COUNT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if len(edges):
    edges[EDGE_SCORE_COLUMN].hist(ax=axes[0], bins=30)
axes[0].set_title(f"{EDGE_SCORE_COLUMN} for kept edges")
axes[0].set_xlabel(EDGE_SCORE_COLUMN)

if len(edges):
    edges["shared_leaders"].hist(ax=axes[1], bins=30)
axes[1].set_title("Shared leaders for kept edges")
axes[1].set_xlabel("shared leaders")

plt.tight_layout()

## Cluster The Overlap Graph Into Provisional Neighborhoods

Once the graph is built, the notebook groups connected singings into provisional neighborhoods. For now, a neighborhood is a connected component in the pruned weighted graph: if you can follow surviving edges from one singing to another, they land in the same component.

This is intentionally simple and inspectable. The tradeoff is that one bridge edge can connect two groups that might feel separate historically, so later cells inspect ties, labels, and geography to see whether the clusters make sense. Small-region merge settings can also consolidate tiny components when their strongest ties are convincing.


In [ ]:
def connected_components(nodes, edge_frame):
    adjacency = defaultdict(set)
    for node in nodes:
        adjacency[node]
    for row in edge_frame.itertuples(index=False):
        adjacency[row.singing_a].add(row.singing_b)
        adjacency[row.singing_b].add(row.singing_a)

    seen = set()
    components = []
    for start in sorted(adjacency):
        if start in seen:
            continue
        queue = deque([start])
        seen.add(start)
        component = []
        while queue:
            node = queue.popleft()
            component.append(node)
            for neighbor in sorted(adjacency[node] - seen):
                seen.add(neighbor)
                queue.append(neighbor)
        components.append(component)
    return components


components = connected_components(eligible_singings.index, edges)
multi_singing_components = [component for component in components if len(component) > 1]
print(f"components with at least two singings: {len(multi_singing_components):,}")
print(f"isolated eligible singings: {sum(1 for component in components if len(component) == 1):,}")

In [ ]:
seed_components = sorted(components, key=lambda c: (-len(c), c[0]))
seed_region_ids = {tuple(component): f"S{index:03d}" for index, component in enumerate(seed_components, start=1)}

seed_region_members = {}
seed_region_leader_sets = {}
seed_region_rows = []

for component in seed_components:
    seed_region_id = seed_region_ids[tuple(component)]
    region_leaders = set().union(*(leader_sets[singing] for singing in component))
    internal_edges = edges[edges["singing_a"].isin(component) & edges["singing_b"].isin(component)]
    seed_region_members[seed_region_id] = list(component)
    seed_region_leader_sets[seed_region_id] = region_leaders
    for singing in component:
        stats = eligible_singings.loc[singing]
        seed_region_rows.append(
            {
                "seed_region_id": seed_region_id,
                "singing": singing,
                "seed_region_size": len(component),
                "seed_region_distinct_leaders": len(region_leaders),
                "seed_internal_edges": len(internal_edges),
                "seed_mean_internal_weighted_jaccard": internal_edges["weighted_jaccard"].mean() if len(internal_edges) else np.nan,
                "mapped_minutes": int(stats["mapped_minutes"]),
                "distinct_leaders": int(stats["distinct_leaders"]),
                "first_year": int(stats["first_year"]) if pd.notna(stats["first_year"]) else np.nan,
                "last_year": int(stats["last_year"]) if pd.notna(stats["last_year"]) else np.nan,
            }
        )

seed_region_stats = pd.DataFrame(seed_region_rows).groupby("seed_region_id").agg(
    seed_region_size=("singing", "count"),
    seed_mapped_minutes=("mapped_minutes", "sum"),
)

current_region_members = {region_id: list(members) for region_id, members in seed_region_members.items()}
current_region_leader_sets = {region_id: set(leaders) for region_id, leaders in seed_region_leader_sets.items()}
merge_targets = {region_id: region_id for region_id in seed_region_members}
merge_scores = {region_id: np.nan for region_id in seed_region_members}
merge_reasons = {region_id: "seed_region" for region_id in seed_region_members}


def weighted_region_jaccard(left_region_id, right_region_id):
    left_leaders = current_region_leader_sets[left_region_id]
    right_leaders = current_region_leader_sets[right_region_id]
    shared_leaders = left_leaders & right_leaders
    shared_weight = sum(leader_weights.get(leader_id, 0.0) for leader_id in shared_leaders)
    left_weight = sum(leader_weights.get(leader_id, 0.0) for leader_id in left_leaders)
    right_weight = sum(leader_weights.get(leader_id, 0.0) for leader_id in right_leaders)
    weighted_union = left_weight + right_weight - shared_weight
    return shared_weight / weighted_union if weighted_union else 0


merge_audit_rows = []
if MERGE_SMALL_REGIONS:
    small_seed_region_ids = (
        seed_region_stats[seed_region_stats["seed_region_size"] < MERGE_REGIONS_SMALLER_THAN]
        .sort_values(["seed_region_size", "seed_mapped_minutes"], ascending=[True, False])
        .index
        .tolist()
    )
    for source_region_id in small_seed_region_ids:
        if merge_targets[source_region_id] != source_region_id:
            continue
        source_size = len(current_region_members[source_region_id])
        best_target = None
        best_score = -1
        for target_region_id in list(current_region_members):
            if target_region_id == source_region_id:
                continue
            target_size = len(current_region_members[target_region_id])
            if source_size + target_size > MAX_MERGED_REGION_SIZE:
                continue
            if target_size < source_size:
                continue
            score = weighted_region_jaccard(source_region_id, target_region_id)
            if score > best_score:
                best_score = score
                best_target = target_region_id
        if best_target is not None and best_score >= MIN_REGION_MERGE_WEIGHTED_JACCARD:
            current_region_members[best_target].extend(current_region_members[source_region_id])
            current_region_leader_sets[best_target].update(current_region_leader_sets[source_region_id])
            del current_region_members[source_region_id]
            del current_region_leader_sets[source_region_id]
            merge_targets[source_region_id] = best_target
            merge_scores[source_region_id] = best_score
            merge_reasons[source_region_id] = "small_region_best_match"
            merge_audit_rows.append(
                {
                    "source_seed_region_id": source_region_id,
                    "target_region_id": best_target,
                    "source_size": source_size,
                    "target_size_before_merge": target_size,
                    "region_weighted_jaccard": best_score,
                }
            )

for region_id in list(merge_targets):
    while merge_targets[region_id] != merge_targets[merge_targets[region_id]]:
        merge_targets[region_id] = merge_targets[merge_targets[region_id]]

region_rows = []
for row in seed_region_rows:
    seed_region_id = row["seed_region_id"]
    final_region_id = merge_targets[seed_region_id]
    final_members = current_region_members.get(final_region_id, seed_region_members[seed_region_id])
    final_leaders = current_region_leader_sets.get(final_region_id, seed_region_leader_sets[seed_region_id])
    region_rows.append(
        {
            **row,
            "region_id": final_region_id,
            "region_size": len(final_members),
            "region_distinct_leaders": len(final_leaders),
            "merge_score": merge_scores[seed_region_id],
            "merge_reason": merge_reasons[seed_region_id],
        }
    )

region_assignments = pd.DataFrame(region_rows).sort_values(
    ["region_id", "seed_region_id", "mapped_minutes", "distinct_leaders"],
    ascending=[True, True, False, False],
)
leading_region_columns = ["singing", "region_id", "region_size"]
seed_region_columns = [column for column in region_assignments.columns if column.startswith("seed_")]
middle_region_columns = [
    column for column in region_assignments.columns
    if column not in leading_region_columns and column not in seed_region_columns
]
region_assignments = region_assignments[
    leading_region_columns + middle_region_columns + seed_region_columns
]

region_summary = (
    region_assignments.groupby("region_id")
    .agg(
        singings=("singing", "count"),
        seed_regions=("seed_region_id", "nunique"),
        mapped_minutes=("mapped_minutes", "sum"),
        region_distinct_leaders=("region_distinct_leaders", "max"),
        first_year=("first_year", "min"),
        last_year=("last_year", "max"),
    )
)
region_summary = region_summary[region_summary["singings"] > 1].sort_values(
    ["singings", "mapped_minutes"],
    ascending=False,
)

merge_audit = pd.DataFrame(merge_audit_rows).sort_values(
    "region_weighted_jaccard",
    ascending=False,
) if merge_audit_rows else pd.DataFrame()

print(f"small seed-region merges applied: {len(merge_audit):,}")
print(f"multi-singing regions after merge: {len(region_summary):,}")
print(f"isolated eligible singings after merge: {(region_assignments['region_size'] == 1).sum():,}")
region_summary

In [ ]:
# Membership list for the larger merged regions.
region_assignments[region_assignments["region_size"] > 1].head(200)

## Generate Region Labels

The graph produces region IDs, but IDs like `S023` are temporary and can change when parameters change. This section creates human-readable labels from the current region contents.

Some labels are curated using anchor singings, so a familiar name can follow the region containing that anchor even if the numeric region ID changes. When there is no curated label, the notebook falls back to names derived from dominant locations or representative singings.

The goal is not to make the labels perfect. The goal is to make tables and maps readable enough for review.


In [ ]:
# Region ids can change when clustering parameters change, so names are generated
# from the current region contents rather than from a fixed region_id mapping.
US_STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AR": "Arkansas", "AZ": "Arizona", "CA": "California",
    "CO": "Colorado", "GA": "Georgia", "IA": "Iowa", "IL": "Illinois", "IN": "Indiana",
    "KY": "Kentucky", "LA": "Louisiana", "MA": "Massachusetts", "MD": "Maryland", "ME": "Maine",
    "MI": "Michigan", "MN": "Minnesota", "MO": "Missouri", "MS": "Mississippi", "NC": "North Carolina",
    "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York", "OH": "Ohio", "OR": "Oregon",
    "PA": "Pennsylvania", "SC": "South Carolina", "TN": "Tennessee", "TX": "Texas", "VA": "Virginia",
    "VT": "Vermont", "WA": "Washington", "WI": "Wisconsin",
}
# Curated names are keyed by stable canonical singing names. If parameters change,
# the name follows the region containing the anchor singing, not the temporary S### id.
REGION_NAME_BY_ANCHOR_SINGING = {
    "Mt. Zion Memorial": "West Georgia",
    "Denney Memorial": "West Georgia",
    "Holly Springs Church": "West Georgia",
    "Union Musical Convention": "West Georgia",
    "Kermit Adams and Steve Adams Memorial Singing": "Northwest Alabama",
    "Winston County Convention": "Northwest Alabama",
    "Shady Grove Annual Singing": "Northwest Alabama",
    "United Kingdom Sacred Harp Convention": "British Isles and Europe",
    "Oxford Sacred Harp Singing Day": "British Isles and Europe",
    "London Singing": "British Isles and Europe",
    "East Midlands Sacred Harp Convention": "Northern England",
    "South Yorkshire Singing": "Northern England",
    "West Yorkshire Sacred Harp Day": "Northern England",
    "Henagar-Union Convention": "Northeast Alabama",
    "Ivey Memorial Singing": "Northeast Alabama",
    "Lacy Memorial": "Northeast Alabama",
    "Labor Day Singing": "East Alabama and West Georgia",
    "Cane Creek Primitive Baptist Church": "East Alabama and West Georgia",
    "All-Wisconsin Sacred Harp Singing": "Upper Midwest",
    "Midwest Convention": "Upper Midwest",
    "Lincoln's Birthday Singing": "Upper Midwest",
    "New England Convention": "New England and Eastern Canada",
    "Western Massachusetts Sacred Harp Singing Convention": "New England and Eastern Canada",
    "Potomac River Convention": "Mid-Atlantic",
    "James River Convention": "Mid-Atlantic",
    "Pacific Northwest Convention - Washington": "Pacific Northwest",
    "Pacific Northwest Convention - Oregon": "Pacific Northwest",
    "All-California Sacred Harp Singing Convention": "California and Mountain West",
    "Rocky Mountain Sacred Harp Singing Convention": "California and Mountain West",
    "Shady Grove (Keeton Cemetery)": "Northwest Alabama Keeton-Flatwoods",
    "Old Flatwoods Primitive Baptist Church": "Northwest Alabama Keeton-Flatwoods",
    "Cotaco Convention": "Huntsville and Middle Tennessee",
    "Harpeth Valley - Priestley Miller Memorial Singing": "Huntsville and Middle Tennessee",
    "Kentucky State Sacred Harp Singing": "Ohio Valley",
    "Ohio State Convention": "Ohio Valley",
    "Boiling Springs Convention": "East-Central Alabama",
    "Alvis Brothers - B. I. Wood Memorial Singing": "Warrior River, Alabama",
    "State Line Church": "Alabama-Georgia State Line",
    "Oxford City Hall": "Mississippi",
    "Illinois State Sacred Harp Convention": "Indiana and Illinois",
    "Hopewell Primitive Baptist Church": "Cleburne and Calhoun Counties",
    "Maidencreek All Day Singing": "Pennsylvania",
    "Mulberry River Convention": "Lamar and Pickens Counties",
    "Pioneer Day Singing": "Carolinas",
    "New Hope Church": "Cullman County, Alabama",
    "South Georgia Sacred Harp Singing Convention": "Middle and South Georgia",
    "Texas State Convention": "Texas",
    "Reid Memorial": "Walker County, Alabama",
    "County Line Church": "Walker County, Alabama",
    "Seed and Feed Sacred Harp Singing": "Atlanta",
    "Spivey Hall Singing": "Atlanta",
    "Victoria Sacred Harp Singing": "Australia",
    "Louisiana State Sacred Harp Convention": "Louisiana",
    "Savannah Sacred Harp Singing": "Coastal Georgia and South Carolina",
    "Barrett Patton's Christmas Singing": "Birmingham Area",
    "Capitol City Shape Note Singing": "Central Alabama",
    "Northwest Arkansas Shiloh Singing, Spring Session": "Northwest Arkansas",
    "Northwest Pennsylvania Singing": "Western Pennsylvania",
    "Alaska Sacred Harp Singing Convention": "Alaska",
    "Chattanooga Area Sacred Harp Singing": "Chattanooga",
    "Conference for the Society of American Music": "Society for American Music",
    "Cork Quaker House Singing": "Cork",
    "Cornucopia All-Night Singing": "Cornucopia",
    "Coweta County Courthouse Singing": "Coweta County",
    "Darien Primitive Baptist Church Homecoming and Singing": "Darien, Alabama",
    "Duke Memorial Singing": "Duke Memorial",
    "Duluth Singing": "Duluth",
    "Feast and FaSoLa": "Feast and FaSoLa",
    "Houston Sacred Harp Singing Convention": "Houston",
    "Jordan's Chapel Methodist Church": "Jordan's Chapel",
    "Kitchens Memorial": "Kitchens Memorial",
    "Little Vine Primitive Baptist Church - Empire": "Empire, Alabama",
    "Marian Bush Memorial Singing": "Marian Bush Memorial",
    "Memphis Singing": "Memphis",
    "Mississippi College": "Mississippi College",
    "Haynes Creek Church Singing": "East Atlanta",
    "Mountain View Shape Note Gathering": "Mountain View, Arkansas",
    "Mt. Zion Primitive Baptist Church": "Mt. Zion Primitive Baptist",
    "New Providence Primitive Baptist Church": "New Providence, Louisiana",
    "New Year's Eve Singing": "New Year's Eve Singing",
    "Newton County Singing": "Newton County",
    "North Carolina Sacred Harp Convention": "North Carolina",
    "Northern California Regional Sacred Harp Singing": "Northern California",
    "Norumbega Harmony Singing": "Norumbega",
    "Old Songs Book Singing": "Old Songs",
    "Pacific Northwest Convention": "Pacific Northwest Convention",
    "Pikes Peak Sacred Harp Singing": "Pikes Peak",
    "Pratt-Woodard Memorial Singing": "Pratt-Woodard Memorial",
    "Rocky Mountain Convention, Colorado Session": "Colorado Rockies",
    "Sardis On the River": "Sardis On the River",
    "Velton Chafin Memorial Singing": "Velton Chafin Memorial",
}


with sqlite3.connect(DB_PATH) as conn:
    label_minute_locations = pd.read_sql_query(
        """
        select distinct
            mlj.minutes_id,
            l.id as location_id,
            l.city,
            l.county,
            l.state_province,
            l.country,
            l.gps_lat,
            l.gps_long
        from minutes_location_joins mlj
        join locations l on l.id = mlj.location_id
        where l.gps_lat is not null
          and l.gps_long is not null
        """,
        conn,
    )

region_label_locations = (
    mapped_minutes[["minutes_id", "canonical_singing"]]
    .merge(label_minute_locations, on="minutes_id", how="inner")
    .merge(
        region_assignments[["singing", "region_id", "region_size", "mapped_minutes"]].rename(
            columns={"singing": "canonical_singing", "mapped_minutes": "mapped_minutes_total"}
        ),
        on="canonical_singing",
        how="inner",
    )
    .drop_duplicates(["canonical_singing", "minutes_id", "location_id"])
)

def _label_mode_or_blank(values):
    values = values.dropna().astype(str).str.strip()
    values = values[values.ne("") & values.ne("nan")]
    if values.empty:
        return ""
    return values.value_counts().index[0]

singing_label_geo = (
    region_label_locations.groupby(["canonical_singing", "region_id", "region_size"], as_index=False)
    .agg(
        latitude=("gps_lat", "median"),
        longitude=("gps_long", "median"),
        state_province=("state_province", _label_mode_or_blank),
        country=("country", _label_mode_or_blank),
        city=("city", _label_mode_or_blank),
        county=("county", _label_mode_or_blank),
    )
    .rename(columns={"canonical_singing": "singing"})
)

region_label_context = region_assignments.merge(singing_label_geo, on=["singing", "region_id", "region_size"], how="left")

def _clean_values(values):
    values = values.dropna().astype(str).str.strip()
    return values[values.ne("") & values.ne("nan")]

def _top_counts(values, n=4):
    counts = _clean_values(values).value_counts().head(n)
    return "; ".join(f"{key}:{count}" for key, count in counts.items())

def _contains_any(text, needles):
    text = text.lower()
    return any(needle.lower() in text for needle in needles)

def _region_text(group):
    cols = ["singing", "city", "county", "state_province", "country"]
    return " | ".join(" ".join(_clean_values(group[col]).tolist()) for col in cols if col in group)

def _dominant_state(group):
    return _label_mode_or_blank(group["state_province"])

def _dominant_country(group):
    return _label_mode_or_blank(group["country"])

def _median_or_none(values):
    values = values.dropna()
    return float(values.median()) if len(values) else None

def _anchor_region_name(group):
    region_singings = set(group["singing"].dropna().astype(str))
    for anchor, region_name in REGION_NAME_BY_ANCHOR_SINGING.items():
        if anchor in region_singings:
            return region_name, anchor
    return None, ""

def _generated_region_name(group):
    anchor_name, anchor = _anchor_region_name(group)
    if anchor_name:
        return anchor_name, "manual_anchor_singing", anchor

    states = set(_clean_values(group["state_province"]))
    countries = set(_clean_values(group["country"]))
    dominant_state = _dominant_state(group)
    dominant_country = _dominant_country(group)
    lat = _median_or_none(group["latitude"])
    lon = _median_or_none(group["longitude"])

    if dominant_country and dominant_country != "USA":
        if len(countries) >= 3 or {"United Kingdom", "Ireland"}.issubset(countries):
            return "British Isles and Europe", "geography_rule_multi_country", ""
        if dominant_country == "United Kingdom" and dominant_state == "England" and lat is not None and lat >= 52.0:
            return "Northern England", "geography_rule_country_region", ""
        return dominant_country, "geography_rule_country", ""

    if {"WA", "OR"}.issubset(states):
        return "Pacific Northwest", "geography_rule_state_cluster", ""
    if states & {"MA", "VT", "ME", "RI", "NH", "CT"}:
        return "New England and Eastern Canada" if "Canada" in countries else "New England", "geography_rule_state_cluster", ""
    if states & {"IL", "MO", "MN", "WI", "IN", "IA"} and len(states & {"IL", "MO", "MN", "WI", "IN", "IA"}) >= 2:
        return "Upper Midwest", "geography_rule_state_cluster", ""
    if states & {"VA", "NY", "NJ", "PA", "MD"} and len(states & {"VA", "NY", "NJ", "PA", "MD"}) >= 2:
        return "Mid-Atlantic", "geography_rule_state_cluster", ""
    if {"AL", "TN"}.issubset(states):
        return "Huntsville and Middle Tennessee", "geography_rule_state_cluster", ""
    if {"AL", "GA"}.issubset(states):
        if dominant_state == "GA" or (lon is not None and lon <= -84.6):
            return "West Georgia", "geography_rule_state_cluster", ""
        return "East Alabama and West Georgia", "geography_rule_state_cluster", ""
    if {"GA", "SC"}.issubset(states):
        return "Coastal Georgia and South Carolina", "geography_rule_state_cluster", ""
    if {"NC", "SC"}.issubset(states):
        return "Carolinas", "geography_rule_state_cluster", ""
    if {"OH", "KY"}.issubset(states) or {"OH", "MI"}.issubset(states):
        return "Ohio Valley", "geography_rule_state_cluster", ""

    if dominant_state == "AL":
        if lon is not None and lon <= -87.0:
            return "Northwest Alabama", "geography_rule_state_subregion", ""
        if lat is not None and lat >= 34.0 and lon is not None and lon >= -86.3:
            return "Northeast Alabama", "geography_rule_state_subregion", ""
        if lat is not None and lat <= 33.5:
            return "East-Central Alabama", "geography_rule_state_subregion", ""
        return "Central Alabama", "geography_rule_state_subregion", ""
    if dominant_state == "GA":
        if lon is not None and lon <= -84.6:
            return "West Georgia", "geography_rule_state_subregion", ""
        if lat is not None and lat <= 33.2:
            return "Middle and South Georgia", "geography_rule_state_subregion", ""
        return "Georgia", "geography_rule_state", ""
    if dominant_state == "CA":
        return "California", "geography_rule_state", ""
    if dominant_state:
        return US_STATE_NAMES.get(dominant_state, dominant_state), "geography_rule_state", ""
    if dominant_country:
        return dominant_country, "geography_rule_country", ""

    top_singing = group.sort_values(["mapped_minutes", "singing"], ascending=[False, True])["singing"].iloc[0]
    return top_singing, "fallback_top_singing", ""

region_name_rows = []
for region_id, group in region_label_context.groupby("region_id", sort=True):
    by_minutes = group.sort_values(["mapped_minutes", "singing"], ascending=[False, True])
    sample_singings = by_minutes["singing"].head(5).tolist()
    latitude = group["latitude"].dropna()
    longitude = group["longitude"].dropna()
    region_name, name_basis, name_anchor = _generated_region_name(group)
    region_name_rows.append(
        {
            "region_id": region_id,
            "region_name": region_name,
            "region_size": int(group["region_size"].max()),
            "singings_with_location": int(group["latitude"].notna().sum()),
            "dominant_state_province": _label_mode_or_blank(group["state_province"]),
            "dominant_country": _label_mode_or_blank(group["country"]),
            "state_province_counts": _top_counts(group["state_province"]),
            "country_counts": _top_counts(group["country"]),
            "centroid_latitude": latitude.median() if len(latitude) else pd.NA,
            "centroid_longitude": longitude.median() if len(longitude) else pd.NA,
            "sample_singings": " | ".join(sample_singings),
            "name_anchor": name_anchor,
            "name_basis": name_basis,
        }
    )

region_name_assignments = pd.DataFrame(region_name_rows).sort_values("region_id").reset_index(drop=True)
region_name_assignments.head(20)


region_name_by_id = region_name_assignments.set_index("region_id")["region_name"].to_dict()

def region_label(region_id):
    region_name = region_name_by_id.get(region_id)
    if region_name:
        return f"{region_id} - {region_name}"
    return str(region_id)

def region_id_from_label(label):
    label = str(label).strip()
    if " - " in label:
        return label.split(" - ", 1)[0]
    return label

def singing_region_id(singing):
    assignment = region_assignments[region_assignments["singing"].eq(singing)]
    if assignment.empty:
        return None
    return assignment.iloc[0]["region_id"]

def singing_region_label(singing):
    region_id = singing_region_id(singing)
    return region_label(region_id) if region_id is not None else ""

region_assignments_labeled = region_assignments.copy()
region_assignments_labeled["region_name"] = region_assignments_labeled["region_id"].map(region_name_by_id)
region_assignments_labeled["region_label"] = region_assignments_labeled["region_id"].map(region_label)

region_summary_labeled = region_summary.copy()
region_summary_labeled["region_name"] = region_summary_labeled.index.map(region_name_by_id)
region_summary_labeled["region_label"] = region_summary_labeled.index.map(region_label)


## Inspect A Singing Or Region

This inspector is the notebook's close-reading tool. After the graph has made broad clusters, this section lets you pick one singing or one region and examine the details behind it.

The controls set `SELECTED_SINGING` and `SELECTED_REGION`, which later cells reuse for strongest-tie tables and heatmaps. If `ipywidgets` is installed, the notebook shows searchable controls. If not, set the globals directly or call:

```python
SELECTED_SINGING = "Winston County Convention"
SELECTED_REGION = singing_region_id(SELECTED_SINGING)
inspect_singing(SELECTED_SINGING)
inspect_region(SELECTED_REGION)
```

Use this section when a region looks surprising. It helps answer, "Which shared leaders are actually holding this neighborhood together?"


In [ ]:
SELECTED_SINGING = globals().get("SELECTED_SINGING", eligible_singings.index[0] if len(eligible_singings) else "")
SELECTED_REGION = globals().get("SELECTED_REGION", singing_region_id(SELECTED_SINGING) if SELECTED_SINGING else (region_summary.index[0] if len(region_summary) else None))


def _edge_rows_for_singing(edge_frame, singing):
    if edge_frame is None or not len(edge_frame):
        return pd.DataFrame()
    rows = edge_frame[edge_frame["singing_a"].eq(singing) | edge_frame["singing_b"].eq(singing)].copy()
    if len(rows):
        rows["other"] = np.where(rows["singing_a"].eq(singing), rows["singing_b"], rows["singing_a"])
    return rows


def _display_table(frame, columns=None, sort_by=None, ascending=False, limit=15):
    if frame is None or not len(frame):
        print("none")
        return
    view = frame.copy()
    if sort_by is not None and sort_by in view.columns:
        view = view.sort_values(sort_by, ascending=ascending)
    if columns is not None:
        view = view[[column for column in columns if column in view.columns]]
    display(view.head(limit))


def _set_selected_singing(singing):
    global SELECTED_SINGING, SELECTED_REGION
    SELECTED_SINGING = singing
    selected_region = singing_region_id(singing)
    if selected_region is not None:
        SELECTED_REGION = selected_region
    return SELECTED_SINGING, SELECTED_REGION


def _set_selected_region(region_id):
    global SELECTED_REGION
    SELECTED_REGION = region_id_from_label(region_id)
    return SELECTED_REGION


def inspect_singing(singing, top_n=15):
    matches = [name for name in eligible_singings.index if str(singing).lower() in name.lower()]
    if singing not in eligible_singings.index:
        if len(matches) == 1:
            singing = matches[0]
        elif len(matches) > 1:
            print(f"No exact eligible singing named {singing!r}. Possible matches:")
            for match in matches[:25]:
                print(" -", match)
            return
        else:
            print(f"{singing!r} is not an eligible singing.")
            if singing in mapped_support.index:
                print("Mapping support:")
                display(mapped_support.loc[[singing]])
            else:
                raw_matches = [name for name in mapped_support.index if str(singing).lower() in name.lower()]
                if raw_matches:
                    print("Possible mapped names:")
                    for match in raw_matches[:25]:
                        print(" -", match)
            return

    _set_selected_singing(singing)

    print(f"## {singing}")
    region_text = singing_region_label(singing)
    if region_text:
        print(f"Selected region: {region_text}")
    print("\nEligibility summary")
    display(eligible_singings.loc[[singing]])

    assignment = region_assignments_labeled[region_assignments_labeled["singing"].eq(singing)]
    print("\nRegion assignment")
    display(assignment)

    if len(assignment):
        row = assignment.iloc[0]
        if row["region_size"] == 1:
            print("\nDiagnosis: isolated after the current graph and merge settings.")
        elif row["merge_reason"] == "small_region_best_match":
            print(
                f"\nDiagnosis: merged from seed {row['seed_region_id']} into {region_label(row['region_id'])} "
                f"with region-level weighted Jaccard {row['merge_score']:.3f}."
            )
        else:
            print(f"\nDiagnosis: member of seed/merged region {region_label(row['region_id'])} with {int(row['region_size'])} singings.")

    final_edges = _edge_rows_for_singing(edges, singing)
    print("\nFinal kept edges")
    _display_table(
        final_edges,
        columns=["other", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient", "edge_source"],
        sort_by=EDGE_SCORE_COLUMN,
        limit=top_n,
    )

    if "all_scored_edges" in globals():
        scored_rows = _edge_rows_for_singing(all_scored_edges, singing)
    else:
        scored_rows = pd.DataFrame()
    print(f"\nBest scored overlaps with at least {MIN_SHARED_LEADERS} shared leaders")
    _display_table(
        scored_rows,
        columns=["other", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient"],
        sort_by=EDGE_SCORE_COLUMN,
        limit=top_n,
    )

    if "candidate_edges" in globals():
        candidate_rows = _edge_rows_for_singing(candidate_edges, singing)
    else:
        candidate_rows = pd.DataFrame()
    print("\nThreshold-passing candidate edges before top-neighbor/fallback pruning")
    _display_table(
        candidate_rows,
        columns=["other", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient"],
        sort_by=EDGE_SCORE_COLUMN,
        limit=top_n,
    )

    if "top_neighbors" in globals() and len(top_neighbors):
        outgoing = top_neighbors[top_neighbors["source"].eq(singing)].copy()
        incoming = top_neighbors[top_neighbors["target"].eq(singing)].copy()
        print(f"\nTop {NEIGHBORS_PER_SINGING} outgoing neighbors")
        _display_table(
            outgoing,
            columns=["target", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient"],
            sort_by=EDGE_SCORE_COLUMN,
            limit=top_n,
        )
        print(f"\nSingings that rank this in their top {NEIGHBORS_PER_SINGING}")
        _display_table(
            incoming,
            columns=["source", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient"],
            sort_by=EDGE_SCORE_COLUMN,
            limit=top_n,
        )

    if "candidate_edges" in globals() and len(candidate_edges):
        all_candidate_rows = _edge_rows_for_singing(candidate_edges, singing)
        fallback_possible = all_candidate_rows[all_candidate_rows[EDGE_SCORE_COLUMN] >= ISOLATE_MIN_WEIGHTED_JACCARD].copy()
        print(f"\nFallback-eligible candidates at ISOLATE_MIN_WEIGHTED_JACCARD >= {ISOLATE_MIN_WEIGHTED_JACCARD}")
        _display_table(
            fallback_possible,
            columns=["other", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient"],
            sort_by=EDGE_SCORE_COLUMN,
            limit=top_n,
        )


def inspect_region(region_id, top_n=200):
    region_id = region_id_from_label(region_id)
    if region_id not in set(region_assignments["region_id"]):
        matches = [rid for rid in sorted(region_assignments["region_id"].unique()) if str(region_id).lower() in str(rid).lower()]
        print(f"No region {region_id!r}. Matches: {matches[:25]}")
        return
    _set_selected_region(region_id)
    members = region_assignments_labeled[region_assignments_labeled["region_id"].eq(region_id)].copy()
    print(f"## Region {region_label(region_id)}")
    print(f"singings: {len(members):,}")
    if region_id in region_summary_labeled.index:
        print("\nRegion summary")
        display(region_summary_labeled.loc[[region_id]])
    print("\nMembers")
    display(
        members[
            [
                "singing",
                "region_label",
                "seed_region_id",
                "seed_region_size",
                "mapped_minutes",
                "distinct_leaders",
                "merge_score",
                "merge_reason",
                "first_year",
                "last_year",
            ]
        ].head(top_n)
    )
    if "merge_audit" in globals() and len(merge_audit):
        audit = merge_audit[merge_audit["target_region_id"].eq(region_id)]
        if len(audit):
            print("\nSmall-region merges into this region")
            display(audit)


try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    singing_picker = widgets.Combobox(
        options=sorted(eligible_singings.index),
        value=SELECTED_SINGING if SELECTED_SINGING in set(eligible_singings.index) else "",
        placeholder="Type a singing name",
        description="Singing:",
        ensure_option=False,
        layout=widgets.Layout(width="700px"),
    )
    inspect_button = widgets.Button(description="Inspect Singing", button_style="primary")
    region_options = [region_label(region_id) for region_id in sorted(region_assignments["region_id"].unique())]
    selected_region_label = region_label(SELECTED_REGION) if SELECTED_REGION is not None else ""
    region_picker = widgets.Combobox(
        options=region_options,
        value=selected_region_label if selected_region_label in region_options else "",
        placeholder="Type a region id or name",
        description="Region:",
        ensure_option=False,
        layout=widgets.Layout(width="500px"),
    )
    inspect_region_button = widgets.Button(description="Inspect Region")
    output = widgets.Output()

    def _sync_region_picker():
        selected_region_label = region_label(SELECTED_REGION) if SELECTED_REGION is not None else ""
        if selected_region_label in region_options:
            region_picker.value = selected_region_label

    def _on_singing_picker_change(change):
        if change["new"] in set(eligible_singings.index):
            _set_selected_singing(change["new"])
            _sync_region_picker()

    def _on_region_picker_change(change):
        region_id = region_id_from_label(change["new"])
        if region_id in set(region_assignments["region_id"]):
            _set_selected_region(region_id)

    def _on_inspect_singing(_):
        with output:
            clear_output()
            inspect_singing(singing_picker.value)
            _sync_region_picker()

    def _on_inspect_region(_):
        with output:
            clear_output()
            inspect_region(region_picker.value)

    singing_picker.observe(_on_singing_picker_change, names="value")
    region_picker.observe(_on_region_picker_change, names="value")
    inspect_button.on_click(_on_inspect_singing)
    inspect_region_button.on_click(_on_inspect_region)
    display(widgets.VBox([
        widgets.HBox([singing_picker, inspect_button]),
        widgets.HBox([region_picker, inspect_region_button]),
        output,
    ]))
except ImportError:
    print("ipywidgets is not installed. Set SELECTED_SINGING / SELECTED_REGION directly, or call inspect_singing('Name') / inspect_region('S001').")
    print("Examples:")
    print("  SELECTED_SINGING = 'Winston County Convention'")
    print("  SELECTED_REGION = singing_region_id(SELECTED_SINGING)")
    print("  inspect_singing(SELECTED_SINGING)")
    print("  inspect_region(SELECTED_REGION)")


## Inspect The Strongest Ties

This table shows the strongest graph ties for `SELECTED_SINGING`, as set by the inspector above. It is a way to check the evidence behind one singing's neighborhood assignment.

The most useful columns are usually the weighted overlap scores and the shared-leader counts. A high score with several shared leaders is stronger evidence than a weak score based on only one or two overlaps.


In [ ]:
selected_edge_rows = _edge_rows_for_singing(edges, SELECTED_SINGING)

print(f"Strongest kept ties for {SELECTED_SINGING}")
_display_table(
    selected_edge_rows,
    columns=["other", "shared_leaders", "jaccard", "weighted_jaccard", "overlap_coefficient", "edge_source"],
    sort_by=EDGE_SCORE_COLUMN,
    limit=TOP_EDGE_COUNT,
)


In [ ]:
def shared_leader_names(singing_a, singing_b, limit=40):
    shared_ids = sorted(leader_sets[singing_a] & leader_sets[singing_b])
    names = (
        singing_leaders[singing_leaders["leader_id"].isin(shared_ids)][["leader_id", "leader_name"]]
        .drop_duplicates()
        .sort_values("leader_name")
    )
    return names.head(limit)


if len(selected_edge_rows):
    example = selected_edge_rows.sort_values([EDGE_SCORE_COLUMN, "shared_leaders"], ascending=False).iloc[0]
    other_singing = example["other"]
    print(SELECTED_SINGING)
    print(other_singing)
    display(shared_leader_names(SELECTED_SINGING, other_singing))
else:
    print(f"No kept edges for {SELECTED_SINGING!r}.")


## Heatmap For One Region

This heatmap visualizes relationships inside one selected region. It plots the region containing `SELECTED_SINGING`, or `SELECTED_REGION` if you inspected a region directly.

Read it as a compact picture of internal structure. Blocks of darker cells suggest groups of singings with strong mutual overlap. Thin or pale connections may point to boundary cases worth checking by hand.


In [ ]:
REGION_TO_PLOT = singing_region_id(SELECTED_SINGING) or SELECTED_REGION

if REGION_TO_PLOT:
    members = region_assignments.loc[region_assignments["region_id"].eq(REGION_TO_PLOT), "singing"].tolist()
    plot_label = region_label(REGION_TO_PLOT)
    if len(members) > MAX_HEATMAP_MEMBERS:
        print(
            f"{plot_label} has {len(members):,} singings; "
            f"raise MAX_HEATMAP_MEMBERS or choose a smaller region before plotting."
        )
    else:
        matrix = pd.DataFrame(index=members, columns=members, dtype=float)
        for left in members:
            for right in members:
                shared_weight = sum(leader_weights.get(leader_id, 0.0) for leader_id in (leader_sets[left] & leader_sets[right]))
                weighted_union = leader_set_weights[left] + leader_set_weights[right] - shared_weight
                matrix.loc[left, right] = shared_weight / weighted_union if weighted_union else 0

        plt.figure(figsize=(min(16, 0.45 * len(members) + 5), min(14, 0.45 * len(members) + 5)))
        sns.heatmap(matrix, cmap="viridis", vmin=0, vmax=max(MIN_WEIGHTED_JACCARD * 2, matrix.to_numpy().max()), square=True)
        plt.title(f"Leader-overlap weighted Jaccard similarities: {plot_label}")
        plt.tight_layout()
else:
    print("No multi-singing regions with the current thresholds.")


## Optional Export

This section writes the graph results to CSV files when `EXPORT_CSVS = True` in the parameter cell. The exports are useful for mapping, sharing, or using the region assignments in another notebook such as the 1991-new-songs analysis.

Leaving export off is fine while experimenting. Turn it on once the current parameter settings produce a version you want to preserve or compare.


In [ ]:
if EXPORT_CSVS:
    edges.to_csv("singing_neighborhood_edges.csv", index=False)
    region_assignments.to_csv("singing_neighborhood_regions.csv", index=False)
    region_name_assignments.to_csv(REGION_NAMES_CSV, index=False)
    print(f"wrote singing_neighborhood_edges.csv, singing_neighborhood_regions.csv, and {REGION_NAMES_CSV}")
else:
    print("Set EXPORT_CSVS = True to write CSV outputs.")


## Region Geography And Equal-Hex Cartogram

This section adds geography to the graph results. It uses location joins in the SQLite database to estimate a home point for each canonical singing, then draws maps of the provisional regions.

There are two different map ideas here. The geographic scatter keeps approximate latitude and longitude, so it shows where singings are located. The equal-hex cartogram gives every eligible singing one same-sized hex, so dense areas do not visually overpower sparse areas. The hex layout preserves rough geography by snapping singing centroids onto the nearest open cell in a regular hex lattice.

The cartogram is a prototype visual summary, not an official boundary map. It is best used to check whether graph neighborhoods also make geographic sense.


In [ ]:
CARTOGRAM_EXPORT_CSV = True
CARTOGRAM_LOCATION_CSV = Path("singing_region_locations.csv")
CARTOGRAM_EXCLUDE_HIGH_LOCATION_SPREAD = True
CARTOGRAM_MAX_LOCATION_P90_MILES = 75
CARTOGRAM_MAX_LOCATION_MAX_MILES = None  # Example: 250

with sqlite3.connect(DB_PATH) as conn:
    minute_locations = pd.read_sql_query(
        """
        select distinct
            mlj.minutes_id,
            l.id as location_id,
            l.name as location_name,
            l.city,
            l.county,
            l.state_province,
            l.country,
            l.gps_lat,
            l.gps_long
        from minutes_location_joins mlj
        join locations l on l.id = mlj.location_id
        where l.gps_lat is not null
          and l.gps_long is not null
        """,
        conn,
    )

singing_minute_locations = (
    mapped_minutes[["minutes_id", "canonical_singing"]]
    .merge(minute_locations, on="minutes_id", how="inner")
    .merge(
        region_assignments[["singing", "region_id", "region_size", "mapped_minutes"]].rename(
            columns={"singing": "canonical_singing", "mapped_minutes": "mapped_minutes_total"}
        ),
        on="canonical_singing",
        how="inner",
    )
    .drop_duplicates(["canonical_singing", "minutes_id", "location_id"])
)

def _mode_or_blank(values):
    values = values.dropna().astype(str).str.strip()
    values = values[values.ne("") & values.ne("nan")]
    if values.empty:
        return ""
    return values.value_counts().index[0]

singing_geo = (
    singing_minute_locations.groupby(["canonical_singing", "region_id", "region_size"], as_index=False)
    .agg(
        mapped_minutes_total=("mapped_minutes_total", "max"),
        located_minutes=("minutes_id", "nunique"),
        location_count=("location_id", "nunique"),
        latitude=("gps_lat", "median"),
        longitude=("gps_long", "median"),
        mean_latitude=("gps_lat", "mean"),
        mean_longitude=("gps_long", "mean"),
        state_province=("state_province", _mode_or_blank),
        country=("country", _mode_or_blank),
        city=("city", _mode_or_blank),
        county=("county", _mode_or_blank),
    )
    .rename(columns={"canonical_singing": "singing"})
    .sort_values(["region_id", "singing"])
    .reset_index(drop=True)
)

def _haversine_miles(lat1, lon1, lat2, lon2):
    radius_miles = 3958.8
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_miles * np.arcsin(np.sqrt(a))

spread_rows = []
for singing, group in singing_minute_locations.groupby("canonical_singing"):
    median_lat = group["gps_lat"].median()
    median_lon = group["gps_long"].median()
    distances = _haversine_miles(group["gps_lat"], group["gps_long"], median_lat, median_lon)
    spread_rows.append(
        {
            "singing": singing,
            "location_p90_miles": float(np.percentile(distances, 90)),
            "location_max_miles": float(np.max(distances)),
            "location_mean_miles": float(np.mean(distances)),
        }
    )

location_spread = pd.DataFrame(spread_rows)
singing_geo = singing_geo.merge(location_spread, on="singing", how="left")

spread_filter = pd.Series(True, index=singing_geo.index)
if CARTOGRAM_EXCLUDE_HIGH_LOCATION_SPREAD:
    if CARTOGRAM_MAX_LOCATION_P90_MILES is not None:
        spread_filter &= singing_geo["location_p90_miles"].le(CARTOGRAM_MAX_LOCATION_P90_MILES)
    if CARTOGRAM_MAX_LOCATION_MAX_MILES is not None:
        spread_filter &= singing_geo["location_max_miles"].le(CARTOGRAM_MAX_LOCATION_MAX_MILES)

excluded_location_spread = singing_geo[~spread_filter].sort_values(
    ["location_p90_miles", "location_max_miles"], ascending=False
).copy()
singing_geo = singing_geo[spread_filter].copy().reset_index(drop=True)

region_geo_summary = (
    singing_geo.groupby("region_id")
    .agg(
        singings_with_location=("singing", "nunique"),
        region_size=("region_size", "max"),
        located_minutes=("located_minutes", "sum"),
        centroid_latitude=("latitude", "median"),
        centroid_longitude=("longitude", "median"),
        dominant_state=("state_province", _mode_or_blank),
        dominant_country=("country", _mode_or_blank),
    )
    .sort_values(["region_size", "singings_with_location"], ascending=False)
)

if CARTOGRAM_EXPORT_CSV:
    singing_geo.to_csv(CARTOGRAM_LOCATION_CSV, index=False)
    print(f"wrote {CARTOGRAM_LOCATION_CSV}")

print(
    f"singings with region assignments: {region_assignments['singing'].nunique():,}; "
    f"with usable GPS locations after spread filter: {singing_geo['singing'].nunique():,}"
)
print(f"excluded for high location spread: {len(excluded_location_spread):,}")

if len(excluded_location_spread):
    display(excluded_location_spread[[
        "singing", "region_id", "region_size", "located_minutes", "location_count",
        "location_p90_miles", "location_max_miles", "state_province", "country"
    ]].head(30))

region_geo_summary.head(20)


In [ ]:
CARTOGRAM_TOP_REGIONS_TO_COLOR = 10
CARTOGRAM_LABEL_TOP_REGIONS = 10
CARTOGRAM_COUNTRY_FILTER = None  # Example: ["USA"]
CARTOGRAM_REGION_FILTER = None  # Example: ["S001", "S004"]
CARTOGRAM_MIN_REGION_SIZE = 8  # Set to 1 to include singleton regions.
CARTOGRAM_FIGSIZE = (16, 10)
CARTOGRAM_HEX_SIZE = 280

if singing_geo.empty:
    raise ValueError("No singing locations are available for the cartogram.")

cartogram_points = singing_geo.copy()
if CARTOGRAM_COUNTRY_FILTER is not None:
    cartogram_points = cartogram_points[cartogram_points["country"].isin(CARTOGRAM_COUNTRY_FILTER)].copy()
if CARTOGRAM_REGION_FILTER is not None:
    cartogram_points = cartogram_points[cartogram_points["region_id"].isin(CARTOGRAM_REGION_FILTER)].copy()
if CARTOGRAM_MIN_REGION_SIZE is not None:
    cartogram_points = cartogram_points[cartogram_points["region_size"].ge(CARTOGRAM_MIN_REGION_SIZE)].copy()
if cartogram_points.empty:
    raise ValueError("The selected cartogram filters left no singing locations to plot.")

lat0 = np.deg2rad(cartogram_points["latitude"].median())
center_lon = cartogram_points["longitude"].median()
center_lat = cartogram_points["latitude"].median()
cartogram_points["geo_x"] = (cartogram_points["longitude"] - center_lon) * np.cos(lat0)
cartogram_points["geo_y"] = cartogram_points["latitude"] - center_lat

# Snap each singing to the nearest open cell in a regular pointy-top hex lattice.
# This gives every singing equal visual weight while preserving rough geographic placement.
dx = 1.0
dy = np.sqrt(3) / 2
x_min = cartogram_points["geo_x"].min()
y_min = cartogram_points["geo_y"].min()
x_span = max(cartogram_points["geo_x"].max() - x_min, 1e-9)
y_span = max(cartogram_points["geo_y"].max() - y_min, 1e-9)
scale = max(x_span / 34, y_span / 22, 1e-9)
cartogram_points["target_col"] = ((cartogram_points["geo_x"] - x_min) / scale).round().astype(int)
cartogram_points["target_row"] = ((cartogram_points["geo_y"] - y_min) / (scale * dy)).round().astype(int)

occupied = set()
assigned_rows = []
search_radius = int(np.ceil(np.sqrt(len(cartogram_points)))) + 10
assignment_order = cartogram_points.sort_values(
    ["region_size", "mapped_minutes_total", "located_minutes", "singing"],
    ascending=[False, False, False, True],
)

for row in assignment_order.itertuples(index=False):
    target_col = int(row.target_col)
    target_row = int(row.target_row)
    best_cell = None
    best_distance = None
    for radius in range(search_radius + 1):
        for rr in range(target_row - radius, target_row + radius + 1):
            for cc in range(target_col - radius, target_col + radius + 1):
                if max(abs(cc - target_col), abs(rr - target_row)) != radius:
                    continue
                if (cc, rr) in occupied:
                    continue
                cell_x = cc + 0.5 * (rr % 2)
                cell_y = rr * dy
                target_x = target_col + 0.5 * (target_row % 2)
                target_y = target_row * dy
                distance = (cell_x - target_x) ** 2 + (cell_y - target_y) ** 2
                if best_distance is None or distance < best_distance:
                    best_cell = (cc, rr, cell_x, cell_y)
                    best_distance = distance
        if best_cell is not None:
            break
    if best_cell is None:
        raise RuntimeError("Could not find an open hex cell; increase search_radius.")
    cc, rr, cell_x, cell_y = best_cell
    occupied.add((cc, rr))
    assigned_rows.append((row.singing, cc, rr, cell_x, cell_y, np.sqrt(best_distance)))

hex_assignments = pd.DataFrame(
    assigned_rows,
    columns=["singing", "hex_col", "hex_row", "hex_x", "hex_y", "hex_displacement"],
)
cartogram_points = cartogram_points.merge(hex_assignments, on="singing", how="left")

region_order = [
    region_id
    for region_id in region_geo_summary.index.tolist()
    if region_id in set(cartogram_points["region_id"])
]
top_regions = region_order[:CARTOGRAM_TOP_REGIONS_TO_COLOR]
palette = sns.color_palette("tab20", 20) + sns.color_palette("Set3", 12) + sns.color_palette("Dark2", 8)
region_colors = {
    region_id: palette[i % len(palette)]
    for i, region_id in enumerate(top_regions)
}
other_color = (0.72, 0.72, 0.72)
cartogram_points["plot_color"] = cartogram_points["region_id"].map(region_colors).apply(
    lambda color: color if isinstance(color, tuple) else other_color
)

print(
    f"cartogram singings: {len(cartogram_points):,}; "
    f"regions: {cartogram_points['region_id'].nunique():,}; "
    f"minimum region size: {CARTOGRAM_MIN_REGION_SIZE}"
)
cartogram_points[[
    "singing", "region_id", "region_size", "hex_col", "hex_row", "hex_displacement",
    "latitude", "longitude", "state_province", "country", "located_minutes", "mapped_minutes_total"
]].sort_values(["region_id", "singing"]).head(20)


### Geographic Scatter

This plot shows the estimated geographic location of each eligible canonical singing. Points are colored by the provisional region assigned from shared-leader overlap.

Use this view to check whether the graph result makes geographic sense. Nearby points with the same color suggest a region that lines up with place. A same-colored point far away from the rest may be a travel, data, or clustering case worth inspecting.


In [ ]:
SCATTER_FIGSIZE = (12, 7)
SCATTER_POINT_SIZE = 42

fig, ax = plt.subplots(figsize=SCATTER_FIGSIZE, constrained_layout=True)
ax.scatter(
    cartogram_points["longitude"],
    cartogram_points["latitude"],
    s=SCATTER_POINT_SIZE,
    c=cartogram_points["plot_color"],
    marker="h",
    linewidths=0.25,
    edgecolors="white",
    alpha=0.88,
)
ax.set_title("Singing home points by region")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_aspect("equal", adjustable="box")
plt.show()


### Equal-Hex Cartogram

This cartogram gives every eligible singing the same visual weight by placing each one in a same-sized hex cell. That means a small rural singing is not hidden by a dense cluster of nearby singings, and a crowded area becomes easier to read.

Because the hexes are snapped to a grid, exact geography is simplified. Read this as a neighborhood diagram with rough placement, not as a precise map.


In [ ]:
CARTOGRAM_FIGSIZE = (20, 14)
CARTOGRAM_HEX_SIZE = 760
CARTOGRAM_LEGEND_COUNT = 18
CARTOGRAM_LABEL_FONT_SIZE = 9

fig, ax = plt.subplots(figsize=CARTOGRAM_FIGSIZE, constrained_layout=True)
ax.scatter(
    cartogram_points["hex_x"],
    cartogram_points["hex_y"],
    s=CARTOGRAM_HEX_SIZE,
    c=cartogram_points["plot_color"],
    marker="h",
    linewidths=0.6,
    edgecolors="white",
)
ax.set_title("Equal-hex singing cartogram")
ax.set_aspect("equal", adjustable="box")
ax.invert_yaxis()
ax.axis("off")

label_regions = region_order[:CARTOGRAM_LABEL_TOP_REGIONS]
label_positions = (
    cartogram_points[cartogram_points["region_id"].isin(label_regions)]
    .groupby("region_id")
    .agg(hex_x=("hex_x", "median"), hex_y=("hex_y", "median"), region_size=("region_size", "max"))
)
for region_id, row in label_positions.iterrows():
    ax.text(
        row["hex_x"],
        row["hex_y"],
        region_id,
        ha="center",
        va="center",
        fontsize=CARTOGRAM_LABEL_FONT_SIZE,
        weight="bold",
        color="black",
        bbox={"boxstyle": "round,pad=0.13", "facecolor": "white", "edgecolor": "none", "alpha": 0.68},
    )

legend_handles = [
    plt.Line2D([0], [0], marker="h", color="w", markerfacecolor=region_colors[r], markersize=12, label=r)
    for r in top_regions[:CARTOGRAM_LEGEND_COUNT]
]
if len(region_order) > CARTOGRAM_TOP_REGIONS_TO_COLOR:
    legend_handles.append(
        plt.Line2D([0], [0], marker="h", color="w", markerfacecolor=other_color, markersize=12, label="other regions")
    )
ax.legend(
    handles=legend_handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
    frameon=True,
    title="Largest regions",
)
plt.show()


## Next Ideas

The current notebook is designed for iteration. The best next step is usually to change one assumption, rerun the notebook, and compare whether the regions become more or less convincing.

Ideas to try:

- Tune `MIN_MAPPED_MINUTES` to decide how much yearly support a singing needs before it participates.
- Tune `NEIGHBORS_PER_SINGING` and `MIN_WEIGHTED_JACCARD` to trade off between larger regions and isolated singings.
- Tune `MERGE_REGIONS_SMALLER_THAN`, `MIN_REGION_MERGE_WEIGHTED_JACCARD`, and `MAX_MERGED_REGION_SIZE` to consolidate tiny regions without recreating one overly connected graph.
- Compare these singing-level neighborhoods with geography once canonical singing locations are settled.
- Try community detection on this pruned graph if connected components still merge too much.
